[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-0/lab-0.2-spec-sheet.ipynb)

# LAB·0.2 · Spec sheet from parts

**Hardware:** any machine. Nothing here touches an accelerator, and nothing imports past the standard library.

A headline FLOPS number is a multiplication somebody already did. Count the multiply-accumulate cells on the chip, allow two operations per cell per cycle, multiply by a clock, and the vendor's figure should come back out. This lab runs that multiplication down the TPU generations and across two GPU dies, then does the memory side, where the arithmetic is division instead.

Some of the runs land on the published number to three digits. Two of them refuse to close at all, and those two are the reason the lab exists. A derivation that fails and names the input it cannot source is worth more than one that quietly picks the number that made it work, so the failures are collected as findings rather than raised as errors.

Every constant carries its source in the comment above it. The lessons behind them are `/s/machine/die-and-reticle`, `/s/machine/inside-the-sm` and `/l/tpu/tensorcore-complex`; the readings on those pages are the primary documents.

In [ ]:
# The frame. One function reconciles a derived number against a published
# one, one collects what refuses to reconcile. A derivation "closes" when it
# lands inside the tolerance; anything else is recorded and carried to the
# end of the notebook instead of being explained away here.
import json

RESULTS = {"notebook": "spec-sheet", "derivations": [], "findings": []}


def reconcile(name, derived, claimed, unit, tol=0.005, note=""):
    gap = derived / claimed - 1.0
    verdict = "closes" if abs(gap) <= tol else "OPEN"
    RESULTS["derivations"].append(
        {"name": name, "derived": round(derived, 4), "claimed": claimed,
         "unit": unit, "gap_pct": round(gap * 100, 3), "verdict": verdict}
    )
    print(f"{name:<38} {derived:>9.4g} vs {claimed:>7.4g} {unit:<7}"
          f" {gap * 100:+7.3f}%  {verdict}"
          + (f"   ({note})" if note else ""))
    return gap


def finding(key, text):
    RESULTS["findings"].append({"id": key, "text": text})
    print(f"  finding recorded · {key}")

## Part 1 · The TPU, where the parts are printed

v4 is the last TPU with a paper behind it. Section 2 of the 2023 ISCA paper names every compute block on the die, Table 4 prints the clock those blocks run at, and both inputs to the multiplication therefore come out of one document. v1 tells the same story eight years earlier with a single array instead of eight.

In [ ]:
# Peak FLOPs from parts: MAC cells x 2 ops per cell per cycle x clock. The
# two is one multiply and one add, and it is the only term here that is not
# looked up.
OPS_PER_MAC = 2


def peak_tflops(macs, clock_hz):
    return macs * OPS_PER_MAC * clock_hz / 1e12


# v1: one 256x256 systolic array.          ISCA 2017, section 2
# clock 700 MHz, published 92 TOPS.        ISCA 2017, Table 2
v1_macs = 256 * 256
reconcile("TPU v1 peak (int8)", peak_tflops(v1_macs, 700e6), 92.0, "TOPS")

# v4: 2 TensorCores x 4 MXUs, each 128x128. ISCA 2023, section 2
# clock 1050 MHz, published 275 TFLOPS.     ISCA 2023, Table 4
v4_macs = 2 * 4 * 128 * 128
v4_peak = peak_tflops(v4_macs, 1050e6)
reconcile("TPU v4 peak (bf16)", v4_peak, 275.0, "TFLOPS")

# One more zoom, same multiplication: a full v4 pod is 4096 chips, and
# Google advertises 1.1 exaflops for it. Two significant figures on the
# claim, so the tolerance has to be loose enough to read them.
# Google Cloud TPU v4 page, via /l/tpu/tensorcore-complex
reconcile("TPU v4 pod, 4096 chips", v4_peak * 4096 / 1e6, 1.1, "EFLOPS",
          tol=0.05, note="advertised to two figures")

print()
print(f"v1 and v4 hold the same cells twice over: {v1_macs} then {v4_macs}")

## The clock you cannot look up

Google stops printing clocks after v4. The v5p page gives two TensorCores and a peak of 459 TFLOPS bf16, and no MXU count per core, so the layout has to be read across from v4 and v5e. That leaves one public figure for the clock and it is secondary. The test of a secondary number is whether it reproduces the vendor's own, and this one does, so the derivation closes with its label still attached.

In [ ]:
# v5p: 2 TensorCores from Google's v5p page; 4 MXUs per core is read across
# from v4 and v5e, which are both documented at four. Same cell count as v4.
v5p_macs = 2 * 4 * 128 * 128

# 1.75 GHz is the scaling book, which is SECONDARY. Google publishes no TPU
# clock after v4, so this is the only public figure that can be run forward.
reconcile("TPU v5p peak (bf16), secondary clock",
          peak_tflops(v5p_macs, 1.75e9), 459.0, "TFLOPS",
          note="clock is secondary, not a vendor line")

# The derivation also runs backwards. Whenever a vendor publishes a peak and
# enough geometry, solve for the constant they withheld and judge the
# third-party claim by whether it lands where the arithmetic says it must.
implied_v5p = 459e12 / (v5p_macs * OPS_PER_MAC)
print(f"v5p clock implied by the published peak: {implied_v5p / 1e9:.4f} GHz")
print(f"same cells as v4 ({v5p_macs}), {459.0 / 275.0:.2f}x the FLOPs, so the "
      "difference is clock and nothing else")

finding("tpu-v5p-secondary",
        "v5p closes to three digits, but only against a clock no vendor "
        "document prints. Label it closed-against-secondary, not closed.")

## Where the arithmetic stops closing

Trillium breaks the pattern. Google's v6e page gives one TensorCore carrying two matrix-multiply units and a peak of 918 TFLOPS bf16, and the architecture page gives the v6e MXU as 256x256. Multiply the three of them, solve for the clock the fourth number would need, and look at what comes out.

In [ ]:
# v6e: 1 TensorCore x 2 MXUs (Google v6e page), each 256x256 (Cloud TPU
# architecture page). Published peak 918 TFLOPS bf16 (Google v6e page).
v6e_macs = 1 * 2 * 256 * 256
V6E_CLAIM = 918e12
implied_v6e = V6E_CLAIM / (v6e_macs * OPS_PER_MAC)

# yardsticks: the fastest clock Google itself has printed is v4 at 1050 MHz,
# and the fastest figure in the public record of any kind is 1.75 GHz for
# v5p (scaling book, secondary).
print(f"v6e cells from the published shape: {v6e_macs}")
print(f"implied clock: {implied_v6e / 1e9:.2f} GHz")
print(f"  against v4, printed:      {1050e6 / 1e9:.2f} GHz "
      f"({implied_v6e / 1050e6:.1f}x)")
print(f"  against v5p, secondary:   {1.75:.2f} GHz "
      f"({implied_v6e / 1.75e9:.1f}x)")
print()

# Two ways to make it close, neither of them evidence.
rescue_macs = 4 * 256 * 256          # assume four MXUs; Google's page says two
print(f"with 4 MXUs instead of 2:  {V6E_CLAIM / (rescue_macs * OPS_PER_MAC) / 1e9:.2f}"
      " GHz, which matches v5p exactly and contradicts the page")

# The same architecture page puts an MXU at 16K multiply-accumulate ops per
# cycle. 16K is 16,384, which is 128x128, not 256x256.
alt_macs = 2 * 16 * 1024
print(f"with the page's own 16K MACs per MXU: "
      f"{V6E_CLAIM / (alt_macs * OPS_PER_MAC) / 1e9:.1f} GHz, worse")
print(f"16K = {16 * 1024} = {int((16 * 1024) ** 0.5)}x{int((16 * 1024) ** 0.5)}, "
      "so the page disagrees with itself about the MXU shape")
print()

# Run the derivation the only honest way left: published shape, and the
# fastest clock anywhere in the public record. It comes up half short.
reconcile("TPU v6e peak at the fastest public clock",
          peak_tflops(v6e_macs, 1.75e9), 918.0, "TFLOPS",
          note="1.75 GHz is v5p's, and no v6e clock exists")

finding("tpu-v6e-open",
        f"The published v6e shape (2 MXUs at 256x256) and the published 918 "
        f"TFLOPS need {implied_v6e / 1e9:.2f} GHz, which no TPU approaches. "
        "Google publishes no v6e clock, and its architecture page gives an "
        "MXU MAC count that fits 128x128 rather than 256x256, so one input "
        "is wrong and nothing public says which.")

## Part 2 · The GPU, where the clock is missing instead

NVIDIA prints the parts list in far more detail than Google and leaves the clock out. Two things have to be settled before any of it can be multiplied: which harvest of the die you were shipped, and whether the rate on the page assumes sparsity.

In [ ]:
# Three chips on one page. H100 whitepaper p.18: the full GH100, the SXM5
# part in the rack, and the PCIe part.
SM_COUNTS = {"GH100 full": 144, "H100 SXM5": 132, "H100 PCIe": 114}
FP32_PER_SM = 128   # whitepaper Table 3: 128 FP32 cores per SM
FP64_PER_SM = 64    # 64 FP64 cores per SM, excluding the Tensor Cores

for sku, sms in SM_COUNTS.items():
    print(f"{sku:<12} {sms:>4} SMs   {sms * FP32_PER_SM:>6} FP32 cores")
print(f"starting from the full die for a part that ships 132 SMs is high by "
      f"{(144 / 132 - 1) * 100:.0f}%")
print()

# The shipping datasheet prints BF16 Tensor Core as 1,979 TFLOPS with an
# asterisk, and the asterisk resolves to "With sparsity": 2:4 structured,
# which the hardware skips through at twice the dense rate. Dense is half.
BF16_SPARSE = 1979.0
BF16_DENSE = BF16_SPARSE / 2
print(f"datasheet BF16 Tensor Core: {BF16_SPARSE} TFLOPS with sparsity")
print(f"dense, the only number you may multiply against: {BF16_DENSE} TFLOPS")

## Pin the constant where the clock is published

FLOPs per SM per clock through the Tensor Cores is the constant nobody publishes. The A100 table prints its own boost clock, so pin the constant on that chip and carry it to Hopper across one documented sentence: Hopper's Tensor Cores deliver twice the A100 SM's matrix-multiply-accumulate rate per clock, on equivalent data types.

In [ ]:
# A100 is the chip whose clock is printed. Whitepaper Table 3, p.39:
# 108 SMs, 1410 MHz boost, FP16 Tensor 312/624 (dense/sparse).
A100_SMS, A100_CLOCK, A100_FP16_DENSE = 108, 1.41e9, 312e12
a100_const = A100_FP16_DENSE / (A100_SMS * A100_CLOCK)
print(f"solve for the constant: {a100_const:.1f} FLOP/SM/clk -> call it 2048")
reconcile("A100 FP16 Tensor, dense", A100_SMS * 2048 * A100_CLOCK / 1e12,
          312.0, "TFLOPS")

# One sentence carries it to Hopper: "2x the MMA computational rates of the
# A100 SM on equivalent data types" (whitepaper p.22), per SM and per clock.
H100_PER_SM = 2048 * 2
H100_SMS = SM_COUNTS["H100 SXM5"]
h100_flops_per_clk = H100_SMS * H100_PER_SM
print(f"\nH100 SXM5: {H100_SMS} SMs x {H100_PER_SM} FLOP/SM/clk "
      f"= {h100_flops_per_clk:,} FLOP/clk")

clock_dense = BF16_DENSE * 1e12 / h100_flops_per_clk
clock_sparse = BF16_SPARSE * 1e12 / h100_flops_per_clk
print(f"clock implied by the dense rate:   {clock_dense / 1e9:.3f} GHz")
print(f"clock implied by the starred rate: {clock_sparse / 1e9:.3f} GHz  "
      "<- what the sparsity trap costs you")
print("third-party databases list 1830 MHz boost for H100 SXM5, which agrees;"
      "\nread that as a check on the arithmetic, not as the missing line")

## The rates that ask for a different clock

Run the same division on the rows that never go through a Tensor Core. Printed rates are rounded, so each one covers a band of clocks rather than a point, and the bands decide whether the spread is rounding or something else.

In [ ]:
# The datasheet rows that do not go through a Tensor Core, and the whitepaper
# preliminary column for the same part. FP32 is 128 lanes x 2 per SM, FP64 is
# 64 x 2, both from whitepaper Table 3.
fp32_per_clk = H100_SMS * FP32_PER_SM * 2
fp64_per_clk = H100_SMS * FP64_PER_SM * 2


def clock_band(printed, per_clk):
    "A printed rate is rounded, so it covers every clock in a band."
    lo = (printed - 0.5) * 1e12 / per_clk
    hi = (printed + 0.5) * 1e12 / per_clk
    return lo / 1e9, hi / 1e9


rows = [
    # rate on the page, TFLOPS, FLOPs per clock, source
    ("datasheet BF16 Tensor, dense", BF16_DENSE, h100_flops_per_clk, "nvidia.com H100"),
    ("datasheet FP32", 67.0, fp32_per_clk, "nvidia.com H100"),
    ("datasheet FP64", 34.0, fp64_per_clk, "nvidia.com H100"),
    ("preliminary FP32", 60.0, fp32_per_clk, "whitepaper Table 3"),
    ("preliminary FP64", 30.0, fp64_per_clk, "whitepaper Table 3"),
    ("preliminary FP16 Tensor, dense", 1000.0, h100_flops_per_clk, "whitepaper Table 3"),
]
for name, rate, per_clk, src in rows:
    lo, hi = clock_band(rate, per_clk)
    print(f"{name:<32} {rate:>7.1f} TF / {per_clk:>7,} = "
          f"{rate * 1e12 / per_clk / 1e9:.3f} GHz   band {lo:.3f}-{hi:.3f}  [{src}]")

# One row cannot be run through this arithmetic at all. FP64 Tensor Core
# prints 67 TFLOPS, the FP32 number over again, and NVIDIA never prints an
# FP64 rate per clock through the Tensor Cores.
print(f"\n{'datasheet FP64 Tensor Core':<32} {67.0:>7.1f} TF / "
      f"{'not published':>13} = cannot be derived")

# Do the two non-Tensor bands overlap, and does the Tensor clock sit in them?
lo32, hi32 = clock_band(67.0, fp32_per_clk)
lo64, hi64 = clock_band(34.0, fp64_per_clk)
overlap = (max(lo32, lo64), min(hi32, hi64))
print(f"\nFP32 and FP64 agree on a band: {overlap[0]:.3f}-{overlap[1]:.3f} GHz")
print(f"the Tensor rows demand {clock_dense / 1e9:.3f} GHz, which is "
      f"{'inside' if overlap[0] <= clock_dense / 1e9 <= overlap[1] else 'outside'}"
      " that band")
print("printed clock, either document: none. The whitepaper says 'Not "
      "Finalized' and the datasheet page lists no clock at all.")

# Assume the whole page is quoted at one clock, take the Tensor rows' clock,
# and predict the FP32 row from it. The miss is the size of the problem.
print()
reconcile("H100 FP32 at the Tensor rows' clock",
          fp32_per_clk * clock_dense / 1e12, 67.0, "TFLOPS",
          note="one page, one clock, assumed")

finding("h100-three-clocks",
        f"One page implies three clocks. The Tensor rows give "
        f"{clock_dense / 1e9:.3f} GHz, FP32 and FP64 agree on a band of "
        f"{overlap[0]:.3f} to {overlap[1]:.3f} GHz once rounding is allowed, "
        "and NVIDIA prints no clock in either document. A peak-FLOPS table "
        "is not necessarily quoted at one clock.")
finding("h100-fp64-tensor",
        "The FP64 Tensor Core row prints 67 TFLOPS and no FLOPs-per-clock "
        "figure exists for it, so its divisor can only be inferred from the "
        "doubling against plain FP64. Recorded as unavailable, not filled in.")

## Part 3 · The memory side, where the division closes

Bandwidth divides as cleanly as FLOPS multiply, with one caveat worth stating before the numbers. A per-stack figure is itself a division of the published total, so multiplying it back is a tautology and proves nothing. The check with content in it compares the per-pin rate against what the JEDEC standard allows.

In [ ]:
# Per part: stacks, capacity, bandwidth. All three totals are vendor lines;
# the per-stack columns below are division. H100 from the whitepaper p.18 and
# the shipping datasheet; B200 and B300 from NVIDIA's Blackwell material.
PARTS = [
    # name, stacks, GB total, TB/s total, bits per stack
    ("H100 SXM5, HBM3", 5, 80, 3.35, 1024),
    ("B200, HBM3e", 8, 192, 8.0, 1024),
    ("B300, HBM3e 12-Hi", 8, 288, 8.0, 1024),
]
# JEDEC HBM3: 6.4 Gb/s per pin, 819 GB/s per device.  JESD238 press release
JEDEC_PIN_GBPS, JEDEC_DEVICE_GBS = 6.4, 819.0

for name, stacks, gb, tbs, bits in PARTS:
    per_stack_gbs = tbs * 1e3 / stacks
    per_pin = per_stack_gbs * 8 / bits            # GB/s -> Gb/s per pin
    print(f"{name:<20} {stacks} stacks  {gb // stacks:>3} GB and "
          f"{per_stack_gbs:>6.1f} GB/s each  {per_pin:.2f} Gb/s per pin"
          f"  {per_pin / JEDEC_PIN_GBPS * 100:>5.0f}% of the HBM3 ceiling")

print("the two Blackwell parts read past 100% because HBM3 is the wrong "
      "yardstick for them:\nthey carry HBM3e, and needing 7.81 Gb/s per pin "
      "is what the E generation was for")

# The H100's 1024 bits per stack is not assumed: the part carries a 5120-bit
# interface across 10 controllers of 512 bits (whitepaper p.18), which is
# 1024 bits on each of the five stacks.
print(f"\nH100 interface check: 10 x 512 bits / 5 stacks = "
      f"{10 * 512 // 5} bits per stack")
print(f"five H100 stacks at the JEDEC ceiling would deliver "
      f"{5 * JEDEC_DEVICE_GBS / 1e3:.1f} TB/s instead of 3.35")
print("B300 keeps 8 stacks and 8 TB/s at 12-high: capacity comes from "
      "stacking dies, bandwidth from interface width and pin rate")

# The TPU side cannot be started. ISCA 2023 Table 4 prints v4's HBM2 as
# 32 GiB at 1200 GB/s and no stack count, and no later generation publishes
# one either.
print(f"\nTPU v4 HBM2: 32 GiB at 1200 GB/s total, stacks: not published")
for n in (2, 4, 6):
    print(f"  if it were {n} stacks: {1200 / n:>6.1f} GB/s each, "
          f"{1200 / n * 8 / 1024:.2f} Gb/s per pin at 1024 bits")

finding("tpu-hbm-stacks",
        "The GPU memory derivation runs because NVIDIA prints stack count, "
        "capacity, bandwidth and interface width. For every TPU generation "
        "the total is published and the stack count is not, so the same "
        "division cannot be started, only bracketed.")

## What did not close

Five derivations landed on the published number, and one of those five got there only against a clock no vendor prints. Two came back open. Each finding names the input that is missing rather than the arithmetic that failed, and the blob at the end is the record to paste back.

In [ ]:
import textwrap

print("=" * 68)
print("DERIVATIONS")
print("=" * 68)
for d in RESULTS["derivations"]:
    print(f"  {d['verdict']:<7} {d['name']:<38} {d['gap_pct']:+7.3f}%")

print()
print("=" * 68)
print("FINDINGS · gaps that stay open")
print("=" * 68)
for i, f in enumerate(RESULTS["findings"], 1):
    print(f"{i}. {f['id']}")
    print(textwrap.fill(f["text"], width=66, initial_indent="   ",
                        subsequent_indent="   "))

print()
print("=" * 68)
print("SPEC-SHEET RESULTS · paste this whole blob back")
print("=" * 68)
print(json.dumps(RESULTS, indent=1))